# NB07 · Thursday — Leakage Hunt #2: The Too-Good Backtest
**Machine Learning in Finance · Week 7, Thursday**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB07_leakage_hunt.ipynb)

**Finance question:** A classmate's fund earns 90% a year and beats the market almost every month. Should we invest?

**Key idea:** A backtest is only as honest as its features. One number computed with future data turns an ordinary strategy into a fantasy.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## The story
A classmate added one homemade feature to the course model: *"trend: how far the price is above its
12-month moving average"*. Their backtest is spectacular. Your team's job: **find the bug before anyone
puts money in.** Run the cells, then investigate.

In [ ]:
panel = mlfin.make_features()
prices = mlfin.load_prices("M")

# Classmate's feature: price relative to its 12-month moving average ("smoothed trend")
moving_avg = prices.rolling(12, center=True).mean()
trend = prices / moving_avg - 1
long = trend.stack().rename("trend").reset_index()
long.columns = ["date", "ticker", "trend"]
panel = panel.merge(long, on=["date", "ticker"], how="left")

FEATURES = mlfin.DEFAULT_BACKTEST_FEATURES + ["trend"]
leaky = mlfin.walk_forward_backtest(panel, mlfin.default_model(), features=FEATURES, rebalance="Q",
                                    cost_bps=20, verbose=False)
leaky["stats"]

In [ ]:
mlfin.plot_cumulative(leaky, log=True)

## Your investigation
Work through these with your team (an AI assistant is allowed, but check what it says):
1. Compare with NB07: SPY earned about 14% a year. Is this result believable for a 30-stock fund of
   large US companies after costs? Name two numbers in the scorecard that are red flags.
2. Look up what `center=True` does in `rolling(...)`. For the row dated **31 December 2019**, which
   months' prices go into its moving average? Try the scratch cell below.
3. Write one sentence explaining why this feature "knows" where the price is going.
4. Fix the feature in the YOUR TURN cell and re-run. What happens?

In [ ]:
# scratch space: compare a centered and a trailing moving average for one stock
check = pd.DataFrame({"price": prices["AAPL"],
                      "centered 12m avg": prices["AAPL"].rolling(12, center=True).mean(),
                      "trailing 12m avg": prices["AAPL"].rolling(12).mean()})
check.loc["2019-06":"2020-06"].round(2)

In [ ]:
# 🟡 YOUR TURN — fix the moving average so it uses only prices known at each date
CENTER = True   # ← the classmate's choice

moving_avg = prices.rolling(12, center=CENTER).mean()
trend = prices / moving_avg - 1
long = trend.stack().rename("trend").reset_index()
long.columns = ["date", "ticker", "trend"]
fixed_panel = panel.drop(columns="trend").merge(long, on=["date", "ticker"], how="left")
fixed = mlfin.walk_forward_backtest(fixed_panel, mlfin.default_model(), features=FEATURES, rebalance="Q",
                                    cost_bps=20, verbose=False)
fixed["stats"]

## Wrap-up: your first project backtest
Leakage hides in "smoothing", "centering", "filling in gaps" and any step that looks at a whole column at
once. Before you trust a backtest, ask: **on each rebalance date, could I have computed every number the
model used?** Now run `mlfin.walk_forward_backtest` with your team's features (NB07 Step 2) and bring one
chart to show-and-tell, with your answer to: *what is the single biggest reason this backtest might be lying?*

In [ ]:
# your team's first backtest
MY_FEATURES = mlfin.DEFAULT_BACKTEST_FEATURES   # ← your proposal's features
mine = mlfin.walk_forward_backtest(mlfin.make_features(), mlfin.default_model(), features=MY_FEATURES,
                                   rebalance="Q", cost_bps=20, verbose=False)
mlfin.plot_cumulative(mine)
mine["stats"]

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*